# 🛡️ The Sentinel — Master ML Pipeline
### GPU Machine (GT 1060 + 16GB RAM)
**Run every cell top to bottom. Do not skip.**

| Cell | What it does |
|---|---|
| 1 | Imports + directory setup + GPU check |
| 2 | IPHasher transformer |
| 3 | UNSW-NB15 merge & clean |
| 4 | CIC-IDS-2017 merge & clean |
| 5 | Combined dataset |
| 6 | UNSW-NB15 preprocessing pipeline |
| 7 | CIC-IDS-2017 preprocessing pipeline |
| 8 | Feature name extractor |
| 9 | Train + evaluate UNSW-NB15 |
| 10 | Train + evaluate CIC-IDS-2017 |
| 11 | Named feature importance plots |
| 12 | Cross-dataset comparison chart |
| 13 | Latency benchmark |
| 14 | Master summary table |
| 15 | Save inference registry |

In [19]:
# ============================================================
# CELL 1 — Imports, Paths, Directory Setup, GPU Check
# FIXED: Notebook runs from ML/ subfolder, so all paths
# use ../ to reach project root
# ============================================================
import os, gc, glob, json, time, hashlib, warnings, joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# ── Root is one level up from ML/ ──
ROOT = os.path.abspath('..')
print(f'[+] Project root : {ROOT}')
print(f'[+] Notebook dir : {os.getcwd()}')

# ── Data paths ──
UNSW_RAW_DIR        = os.path.join(ROOT, 'Data', 'raw', 'UNSW-NB15')
CIC_RAW_DIR         = os.path.join(ROOT, 'Data', 'raw', 'CIC-IDS-2017', 'TrafficLabelling')
UNSW_CLEAN_PATH     = os.path.join(ROOT, 'Data', 'clean', 'clean_unsw_nb15_all.csv')
CIC_CLEAN_PATH      = os.path.join(ROOT, 'Data', 'clean', 'clean_cic_ids2017.csv')
COMBINED_CLEAN_PATH = os.path.join(ROOT, 'Data', 'clean', 'clean_combined_all.csv')

# ── ML paths ──
UNSW_PIPELINE_PATH     = os.path.join(ROOT, 'ML', 'preprocessing', 'UNSW-NB15', 'preprocessing_pipeline.pkl')
UNSW_FEAT_COLUMNS_PATH = os.path.join(ROOT, 'ML', 'preprocessing', 'UNSW-NB15', 'feature_columns.joblib')
CIC_PIPELINE_PATH      = os.path.join(ROOT, 'ML', 'preprocessing', 'CIC-IDS-2017', 'preprocessing_pipeline.pkl')
CIC_FEAT_COLUMNS_PATH  = os.path.join(ROOT, 'ML', 'preprocessing', 'CIC-IDS-2017', 'feature_columns.joblib')

UNSW_MODEL_DIR      = os.path.join(ROOT, 'ML', 'models', 'UNSW-NB15')
CIC_MODEL_DIR       = os.path.join(ROOT, 'ML', 'models', 'CIC-IDS-2017')
INFERENCE_PATH      = os.path.join(ROOT, 'ML', 'inference_models', 'inference_registry.pkl')

UNSW_PLOT_DIR       = os.path.join(ROOT, 'ML', 'evaluation', 'plots', 'UNSW-NB15')
CIC_PLOT_DIR        = os.path.join(ROOT, 'ML', 'evaluation', 'plots', 'CIC-IDS-2017')
CROSS_PLOT_DIR      = os.path.join(ROOT, 'ML', 'evaluation', 'plots', 'CrossDataset')
LATENCY_PLOT_DIR    = os.path.join(ROOT, 'ML', 'evaluation', 'plots', 'Latency')
METRICS_DIR         = os.path.join(ROOT, 'ML', 'evaluation', 'metrics')

# ── Create all directories ──
for d in [
    os.path.join(ROOT, 'Data', 'clean'),
    UNSW_MODEL_DIR, CIC_MODEL_DIR,
    os.path.join(ROOT, 'ML', 'models', 'Combined'),
    os.path.join(ROOT, 'ML', 'preprocessing', 'UNSW-NB15'),
    os.path.join(ROOT, 'ML', 'preprocessing', 'CIC-IDS-2017'),
    UNSW_PLOT_DIR, CIC_PLOT_DIR, CROSS_PLOT_DIR,
    LATENCY_PLOT_DIR, METRICS_DIR,
    os.path.join(ROOT, 'ML', 'inference_models'),
]:
    os.makedirs(d, exist_ok=True)

# ── Verify raw data exists ──
unsw_files = glob.glob(os.path.join(UNSW_RAW_DIR, 'UNSW-NB15_*.csv'))
cic_files  = glob.glob(os.path.join(CIC_RAW_DIR, '*.csv'))
print(f'[+] UNSW-NB15 raw files found : {len(unsw_files)}')
print(f'[+] CIC-IDS-2017 files found  : {len(cic_files)}')

if len(unsw_files) == 0:
    print(f'❌ UNSW files not found at: {UNSW_RAW_DIR}')
if len(cic_files) == 0:
    print(f'❌ CIC files not found at : {CIC_RAW_DIR}')

# ── GPU check ──
try:
    import xgboost as xgb
    _clf = xgb.XGBClassifier(
        tree_method='gpu_hist', device='cuda',
        n_estimators=2, verbosity=0
    )
    _clf.fit(np.random.rand(100, 5), np.random.randint(0, 2, 100))
    GPU_AVAILABLE   = True
    XGB_TREE_METHOD = 'gpu_hist'
    XGB_DEVICE      = 'cuda'
    print('✅ GPU detected — XGBoost will use CUDA')
except Exception as _e:
    GPU_AVAILABLE   = False
    XGB_TREE_METHOD = 'hist'
    XGB_DEVICE      = 'cpu'
    print(f'⚠️  GPU not available — using CPU')

MODEL_DISPLAY = {
    'random_forest':       'Random Forest',
    'xgboost':             'XGBoost',
    'logistic_regression': 'Logistic Regression'
}

print(f'\n[+] XGBoost: tree_method={XGB_TREE_METHOD}')
print('✅ Cell 1 complete')

[+] Project root : d:\AGENTIC-AI-CYBERSECURITY
[+] Notebook dir : d:\AGENTIC-AI-CYBERSECURITY\ML
[+] UNSW-NB15 raw files found : 4
[+] CIC-IDS-2017 files found  : 8
⚠️  GPU not available — using CPU

[+] XGBoost: tree_method=hist
✅ Cell 1 complete


In [20]:
# ============================================================
# CELL 2 — Import Transformers From Module
# ============================================================
import sys
import os
import importlib.util

# Fix: go one level UP from ML/ to reach project root
_ml_dir      = os.getcwd()                    # D:\AGENTIC-AI-CYBERSECURITY\ML
_project_root = os.path.dirname(_ml_dir)      # D:\AGENTIC-AI-CYBERSECURITY

_transformers_path = os.path.join(
    _project_root, 'ML', 'preprocessing', 'custom_transformers.py'
)

print(f'[+] ML dir       : {_ml_dir}')
print(f'[+] Project root : {_project_root}')
print(f'[+] Looking for  : {_transformers_path}')

if not os.path.exists(_transformers_path):
    raise FileNotFoundError(
        f"custom_transformers.py not found at:\n{_transformers_path}"
    )

_spec   = importlib.util.spec_from_file_location(
    "ML.preprocessing.custom_transformers",
    _transformers_path
)
_module = importlib.util.module_from_spec(_spec)
sys.modules["ML.preprocessing.custom_transformers"] = _module
_spec.loader.exec_module(_module)

IPHasher     = _module.IPHasher
FullPipeline = _module.FullPipeline

print(f'[+] IPHasher     : {IPHasher}')
print(f'[+] FullPipeline : {FullPipeline}')
print(f'[+] Module name  : {FullPipeline.__module__}')
print('✅ Cell 2 complete — verify module name shows ML.preprocessing.custom_transformers')

[+] ML dir       : d:\AGENTIC-AI-CYBERSECURITY\ML
[+] Project root : d:\AGENTIC-AI-CYBERSECURITY
[+] Looking for  : d:\AGENTIC-AI-CYBERSECURITY\ML\preprocessing\custom_transformers.py
[+] IPHasher     : <class 'ML.preprocessing.custom_transformers.IPHasher'>
[+] FullPipeline : <class 'ML.preprocessing.custom_transformers.FullPipeline'>
[+] Module name  : ML.preprocessing.custom_transformers
✅ Cell 2 complete — verify module name shows ML.preprocessing.custom_transformers


In [23]:
# ============================================================
# CELL 3 — UNSW-NB15: Merge & Clean
# Loads UNSW-NB15_1.csv through _4.csv from Data/raw/UNSW-NB15/
# Skips if clean_unsw_nb15_all.csv already exists
# ============================================================
UNSW_COLUMN_NAMES = [
    'srcip','sport','dstip','dsport','proto','state','dur',
    'sbytes','dbytes','sttl','dttl','sloss','dloss','service',
    'Sload','Dload','Spkts','Dpkts','swin','dwin','stcpb','dtcpb',
    'smeansz','dmeansz','trans_depth','res_bdy_len','Sjit','Djit',
    'Stime','Ltime','Sintpkt','Dintpkt','tcprtt','synack','ackdat',
    'is_sm_ips_ports','ct_state_ttl','ct_flw_http_mthd','is_ftp_login',
    'ct_ftp_cmd','ct_srv_src','ct_srv_dst','ct_dst_ltm','ct_src_ltm',
    'ct_src_dport_ltm','ct_dst_sport_ltm','ct_dst_src_ltm',
    'attack_cat','label'
]

if os.path.exists(UNSW_CLEAN_PATH):
    print(f'[+] clean_unsw_nb15_all.csv already exists — skipping merge')
    _df = pd.read_csv(UNSW_CLEAN_PATH, nrows=2)
    if 'dataset_source' not in _df.columns:
        print('[+] Adding dataset_source column to existing file...')
        df_fix = pd.read_csv(UNSW_CLEAN_PATH, low_memory=False)
        df_fix['dataset_source'] = 'UNSW-NB15'
        df_fix.to_csv(UNSW_CLEAN_PATH, index=False)
        del df_fix; gc.collect()
        print('[+] Done.')
    row_count = sum(1 for _ in open(UNSW_CLEAN_PATH)) - 1
    print(f'[+] Existing file rows: {row_count:,}')
else:
    unsw_files = sorted(glob.glob(f'{UNSW_RAW_DIR}/UNSW-NB15_*.csv'))
    print(f'[+] Found {len(unsw_files)} UNSW-NB15 raw files')
    if len(unsw_files) == 0:
        print(f'❌ No files found in {UNSW_RAW_DIR}')
        print('   Expected: UNSW-NB15_1.csv, UNSW-NB15_2.csv, UNSW-NB15_3.csv, UNSW-NB15_4.csv')
    else:
        dfs = []
        for f in unsw_files:
            print(f'    Loading {os.path.basename(f)}...')
            dfs.append(pd.read_csv(
                f, header=None, names=UNSW_COLUMN_NAMES, low_memory=False
            ))
        df = pd.concat(dfs, ignore_index=True)
        del dfs; gc.collect()
        print(f'[+] Combined: {df.shape}')

        df = df.dropna(subset=['label'])
        df = df.drop_duplicates()
        df['attack_cat']     = df['attack_cat'].fillna('Normal').str.strip()
        df['label']          = pd.to_numeric(df['label'], errors='coerce').fillna(0).astype(int)
        df['dataset_source'] = 'UNSW-NB15'

        print(f'[+] After clean: {df.shape}')
        print(f'[+] Labels:\n{df["label"].value_counts()}')
        print(f'[+] Attack cats:\n{df["attack_cat"].value_counts()}')
        df.to_csv(UNSW_CLEAN_PATH, index=False)
        del df; gc.collect()

print(f'✅ Cell 3 complete — UNSW-NB15 clean at {UNSW_CLEAN_PATH}')

[+] clean_unsw_nb15_all.csv already exists — skipping merge
[+] Existing file rows: 2,059,415
✅ Cell 3 complete — UNSW-NB15 clean at d:\AGENTIC-AI-CYBERSECURITY\Data\clean\clean_unsw_nb15_all.csv


In [24]:
# ============================================================
# CELL 4 — CIC-IDS-2017: Merge & Clean
# Path: Data/raw/CIC-IDS-2017/TrafficLabelling_/
# Per-file processing — avoids MemoryError on 3M rows
# encoding=latin-1 — required for Windows-encoded CIC files
# ============================================================
CIC_DROP_COLS = [
    'Flow ID', 'Source IP', 'Destination IP',
    'Source Port', 'Destination Port', 'Timestamp'
]

cic_files = sorted(glob.glob(f'{CIC_RAW_DIR}/*.csv'))
print(f'[+] CIC path: {CIC_RAW_DIR}')
print(f'[+] Found {len(cic_files)} CIC-IDS-2017 files:')
for f in cic_files:
    print(f'    {os.path.basename(f)}')

if len(cic_files) == 0:
    print(f'\n❌ No CSV files found in {CIC_RAW_DIR}')
    print('   Check your folder name — should be TrafficLabelling_ (with underscore)')
else:
    first_file    = True
    total_written = 0

    for f in cic_files:
        fname = os.path.basename(f)
        print(f'\n    Processing {fname}...')

        # latin-1 required — CIC files are Windows-encoded
        df = pd.read_csv(f, encoding='latin-1', low_memory=False)
        df.columns = df.columns.str.strip()

        # Fix infinity/NaN immediately
        df.replace([np.inf, -np.inf], np.nan, inplace=True)

        # Drop leakage columns
        df.drop(columns=CIC_DROP_COLS, errors='ignore', inplace=True)

        if 'Label' not in df.columns:
            print(f'    ⚠️  No Label column — skipping {fname}')
            continue

        df['Label']      = df['Label'].astype(str).str.strip()
        df['attack_cat'] = df['Label'].apply(
            lambda x: 'Normal' if x.upper() == 'BENIGN' else x
        )
        df['label'] = df['Label'].apply(
            lambda x: 0 if x.upper() == 'BENIGN' else 1
        )
        df.drop(columns=['Label'], errors='ignore', inplace=True)

        # Drop columns with more than 50% NaN
        df.dropna(axis=1, thresh=int(len(df) * 0.5), inplace=True)
        df.dropna(how='all', inplace=True)

        # Deduplicate per file — far cheaper than full 3M row dedup
        before = len(df)
        df.drop_duplicates(inplace=True)
        print(f'    Shape: {df.shape} | Dupes removed: {before - len(df):,}')

        df['dataset_source'] = 'CIC-IDS-2017'

        df.to_csv(
            CIC_CLEAN_PATH,
            mode='w' if first_file else 'a',
            header=first_file,
            index=False
        )
        total_written += len(df)
        first_file = False
        del df; gc.collect()
        print(f'    ✅ Written. Total so far: {total_written:,}')

    # Verify
    row_count = sum(1 for _ in open(CIC_CLEAN_PATH)) - 1
    label_counts = {0: 0, 1: 0}
    for chunk in pd.read_csv(
            CIC_CLEAN_PATH, usecols=['label'], chunksize=200_000):
        for k, v in chunk['label'].value_counts().items():
            label_counts[int(k)] = label_counts.get(int(k), 0) + int(v)

    print(f'\n[+] Total rows : {row_count:,}')
    print(f'[+] Normal (0) : {label_counts.get(0,0):,}')
    print(f'[+] Attack (1) : {label_counts.get(1,0):,}')

print(f'\n✅ Cell 4 complete — CIC saved to {CIC_CLEAN_PATH}')

[+] CIC path: d:\AGENTIC-AI-CYBERSECURITY\Data\raw\CIC-IDS-2017\TrafficLabelling
[+] Found 8 CIC-IDS-2017 files:
    Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
    Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
    Friday-WorkingHours-Morning.pcap_ISCX.csv
    Monday-WorkingHours.pcap_ISCX.csv
    Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
    Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
    Tuesday-WorkingHours.pcap_ISCX.csv
    Wednesday-workingHours.pcap_ISCX.csv

    Processing Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...
    Shape: (221290, 80) | Dupes removed: 4,455
    ✅ Written. Total so far: 221,290

    Processing Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv...
    Shape: (119578, 80) | Dupes removed: 166,889
    ✅ Written. Total so far: 340,868

    Processing Friday-WorkingHours-Morning.pcap_ISCX.csv...
    Shape: (176125, 80) | Dupes removed: 14,908
    ✅ Written. Total so far: 516,993

    Processing Monday-WorkingHour

In [25]:
# ============================================================
# CELL 5 — Combined Dataset: UNSW-NB15 + CIC-IDS-2017
# Finds common features between both datasets
# UNSW loaded in RAM, CIC streamed in chunks
# ============================================================
print('[+] Loading UNSW-NB15...')
unsw = pd.read_csv(UNSW_CLEAN_PATH, low_memory=False)
print(f'    Shape: {unsw.shape}')

META_COLS      = ['label', 'attack_cat', 'dataset_source']
unsw_feat_cols = set(unsw.columns) - set(META_COLS)

cic_header    = pd.read_csv(CIC_CLEAN_PATH, nrows=0)
cic_feat_cols = set(cic_header.columns) - set(META_COLS)

common_features = sorted(unsw_feat_cols & cic_feat_cols)
print(f'[+] UNSW features   : {len(unsw_feat_cols)}')
print(f'[+] CIC  features   : {len(cic_feat_cols)}')
print(f'[+] Common features : {len(common_features)}')
if common_features:
    print(f'[+] Common cols     : {common_features}')
else:
    print('[+] No common features — datasets have different schemas')
    print('    Combined file will use meta cols only')
    print('    Cross-dataset evaluation in Cell 12 still works')

use_cols = (common_features + META_COLS) if common_features else META_COLS

# Write UNSW portion first
unsw[use_cols].to_csv(COMBINED_CLEAN_PATH, index=False, mode='w', header=True)
del unsw; gc.collect()
print('[+] UNSW-NB15 written to combined file')

# Stream CIC in chunks
chunks_done = 0
for chunk in pd.read_csv(
        CIC_CLEAN_PATH, usecols=use_cols,
        chunksize=200_000, low_memory=False):
    chunk.to_csv(COMBINED_CLEAN_PATH, index=False, mode='a', header=False)
    chunks_done += 1

row_count = sum(1 for _ in open(COMBINED_CLEAN_PATH)) - 1
print(f'[+] CIC written in {chunks_done} chunks')
print(f'[+] Combined total rows: {row_count:,}')
print(f'✅ Cell 5 complete — saved to {COMBINED_CLEAN_PATH}')

[+] Loading UNSW-NB15...
    Shape: (2059415, 50)
[+] UNSW features   : 47
[+] CIC  features   : 78
[+] Common features : 0
[+] No common features — datasets have different schemas
    Combined file will use meta cols only
    Cross-dataset evaluation in Cell 12 still works
[+] UNSW-NB15 written to combined file
[+] CIC written in 11 chunks
[+] Combined total rows: 4,218,419
✅ Cell 5 complete — saved to d:\AGENTIC-AI-CYBERSECURITY\Data\clean\clean_combined_all.csv


In [26]:
 
# ============================================================
# CELL 6 — UNSW-NB15: Build Preprocessing Pipeline
# NO CHANGES to pipeline logic — only FullPipeline is now
# imported from module instead of defined here
# ============================================================
print('[+] Loading UNSW-NB15...')
df_unsw = pd.read_csv(UNSW_CLEAN_PATH, low_memory=False)
X_unsw  = df_unsw.drop(
    columns=['label', 'attack_cat', 'dataset_source'], errors='ignore'
)
del df_unsw; gc.collect()
print(f'[+] Feature matrix: {X_unsw.shape}')
 
# Save raw feature names before any transformation
unsw_raw_feature_names = X_unsw.columns.tolist()
joblib.dump(unsw_raw_feature_names, UNSW_FEAT_COLUMNS_PATH)
print(f'[+] Raw feature names: {unsw_raw_feature_names}')
 
# Step 1: Hash IP columns first
ip_cols   = [c for c in ['srcip', 'dstip'] if c in X_unsw.columns]
ip_hasher = IPHasher(columns=ip_cols)
X_hashed  = ip_hasher.transform(X_unsw)
del X_unsw; gc.collect()
print(f'[+] IP cols hashed: {ip_cols}')
 
# Step 2: Detect column types ON hashed data
categorical_cols = [
    c for c in X_hashed.select_dtypes(include=['object']).columns
    if c not in ip_cols
]
numeric_cols = X_hashed.select_dtypes(
    include=['int64','float64','int32','float32']
).columns.tolist()
for col in ip_cols:
    if col not in numeric_cols:
        numeric_cols.append(col)
 
print(f'[+] Numeric cols    : {len(numeric_cols)}')
print(f'[+] Categorical cols: {len(categorical_cols)} → {categorical_cols}')
 
# Step 3: Cardinality check — OHE only on low-cardinality cols
safe_categorical = []
high_card_cols_u = []
for col in categorical_cols:
    n = X_hashed[col].nunique()
    if n <= 50:
        safe_categorical.append(col)
        print(f'    ✅ {col}: {n} unique → OHE')
    else:
        high_card_cols_u.append(col)
        X_hashed[col] = pd.factorize(X_hashed[col])[0]
        numeric_cols.append(col)
        print(f'    ⚠️  {col}: {n} unique → factorize (too high for OHE)')
 
categorical_cols = safe_categorical
 
# Step 4: Build transformers
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
transformers = [('num', numeric_transformer, numeric_cols)]
 
if categorical_cols:
    cat_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=True,
            max_categories=20
        ))
    ])
    transformers.append(('cat', cat_transformer, categorical_cols))
 
preprocessor_unsw = ColumnTransformer(
    transformers=transformers, remainder='drop'
)
 
print(f'[+] Fitting pipeline on {len(X_hashed):,} rows...')
preprocessor_unsw.fit(X_hashed)
del X_hashed; gc.collect()
 
# Step 5: Build pipeline using IMPORTED FullPipeline (not defined here)
# FullPipeline is imported from ML.preprocessing.custom_transformers in Cell 2
pipeline_unsw = FullPipeline(ip_hasher, preprocessor_unsw, high_card_cols_u)
joblib.dump(pipeline_unsw, UNSW_PIPELINE_PATH)
 
print(f'✅ Cell 6 complete — pipeline saved to {UNSW_PIPELINE_PATH}')
print(f'   Numeric: {len(numeric_cols)} | OHE: {len(categorical_cols)} | HighCard: {high_card_cols_u}')
print(f'   FullPipeline class: {FullPipeline.__module__}.{FullPipeline.__name__}')
print(f'   ✅ Verify above line shows: ML.preprocessing.custom_transformers.FullPipeline')

[+] Loading UNSW-NB15...
[+] Feature matrix: (2059415, 47)
[+] Raw feature names: ['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm']
[+] IP cols hashed: ['srcip', 'dstip']
[+] Numeric cols    : 41
[+] Categorical cols: 6 → ['sport', 'dsport', 'proto', 'state', 'service', 'ct_ftp_cmd']
    ⚠️  sport: 64600 unique → factorize (too high for OHE)
    ⚠️  dsport: 64630 unique → factorize (too high for OHE)
    ⚠️  proto: 135 unique → factorize (too high for OHE)
    ✅ state: 16 unique → OHE
    ✅ service: 13 unique 

In [27]:
# ============================================================
# CELL 7 — CIC-IDS-2017: Build Preprocessing Pipeline
# IPs already dropped in Cell 4 — no IPHasher needed
# Sample 200K rows for fitting — safe + fast
# ============================================================
print('[+] Loading 200K sample from CIC-IDS-2017...')

chunks, total = [], 0
for chunk in pd.read_csv(CIC_CLEAN_PATH, chunksize=100_000, low_memory=False):
    chunks.append(chunk)
    total += len(chunk)
    if total >= 200_000:
        break

df_sample    = pd.concat(chunks, ignore_index=True).head(200_000)
del chunks; gc.collect()
print(f'[+] Sample shape: {df_sample.shape}')

X_cic_sample = df_sample.drop(
    columns=['label', 'attack_cat', 'dataset_source'], errors='ignore'
)
del df_sample; gc.collect()

# Save named feature columns
cic_raw_feature_names = X_cic_sample.columns.tolist()
joblib.dump(cic_raw_feature_names, CIC_FEAT_COLUMNS_PATH)
print(f'[+] CIC feature names saved: {len(cic_raw_feature_names)}')
print(f'[+] Sample: {cic_raw_feature_names[:8]}')

# Column types
categorical_cols_cic = X_cic_sample.select_dtypes(
    include=['object']
).columns.tolist()
numeric_cols_cic = X_cic_sample.select_dtypes(
    include=['int64','float64','int32','float32']
).columns.tolist()
print(f'[+] Numeric: {len(numeric_cols_cic)} | Categorical: {categorical_cols_cic}')

# Cardinality check
safe_categorical_cic = []
high_card_cols_cic   = []
for col in categorical_cols_cic:
    n = X_cic_sample[col].nunique()
    if n <= 50:
        safe_categorical_cic.append(col)
        print(f'    ✅ {col}: {n} unique → OHE')
    else:
        high_card_cols_cic.append(col)
        X_cic_sample[col] = pd.factorize(X_cic_sample[col])[0]
        numeric_cols_cic.append(col)
        print(f'    ⚠️  {col}: {n} unique → factorize')

categorical_cols_cic = safe_categorical_cic

# Build pipeline
num_transformer_cic = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
transformers_cic = [('num', num_transformer_cic, numeric_cols_cic)]

if categorical_cols_cic:
    cat_transformer_cic = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=True,
            max_categories=20
        ))
    ])
    transformers_cic.append(('cat', cat_transformer_cic, categorical_cols_cic))

preprocessor_cic = ColumnTransformer(
    transformers=transformers_cic, remainder='drop'
)
pipeline_cic = Pipeline([('preprocessor', preprocessor_cic)])

print(f'[+] Fitting CIC pipeline on {len(X_cic_sample):,} rows...')
pipeline_cic.fit(X_cic_sample)
del X_cic_sample; gc.collect()

joblib.dump(pipeline_cic, CIC_PIPELINE_PATH)
print(f'✅ Cell 7 complete — CIC pipeline saved to {CIC_PIPELINE_PATH}')
print(f'   Numeric: {len(numeric_cols_cic)} | OHE: {len(categorical_cols_cic)} | HighCard: {high_card_cols_cic}')

[+] Loading 200K sample from CIC-IDS-2017...
[+] Sample shape: (200000, 81)
[+] CIC feature names saved: 78
[+] Sample: ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min']
[+] Numeric: 78 | Categorical: []
[+] Fitting CIC pipeline on 200,000 rows...
✅ Cell 7 complete — CIC pipeline saved to d:\AGENTIC-AI-CYBERSECURITY\ML\preprocessing\CIC-IDS-2017\preprocessing_pipeline.pkl
   Numeric: 78 | OHE: 0 | HighCard: []


In [28]:
# ============================================================
# CELL 8 — Feature Name Extractor
# Fixes Feature_N generic names → real column names in plots
# ============================================================
def get_feature_names_unsw(preprocessor, num_cols, cat_cols):
    try:
        names = list(num_cols)
        if cat_cols:
            ohe = preprocessor.named_transformers_['cat'].named_steps['encoder']
            names += list(ohe.get_feature_names_out(cat_cols))
        return names
    except Exception as e:
        print(f'⚠️  UNSW name extraction failed: {e}')
        return list(num_cols)

def get_feature_names_cic(pipeline_cic, num_cols, cat_cols):
    try:
        ct    = pipeline_cic.named_steps['preprocessor']
        names = list(num_cols)
        if cat_cols:
            ohe = ct.named_transformers_['cat'].named_steps['encoder']
            names += list(ohe.get_feature_names_out(cat_cols))
        return names
    except Exception as e:
        print(f'⚠️  CIC name extraction failed: {e}')
        return list(num_cols)

unsw_feature_names = get_feature_names_unsw(
    pipeline_unsw.preprocessor, numeric_cols, categorical_cols
)
cic_feature_names = get_feature_names_cic(
    pipeline_cic, numeric_cols_cic, categorical_cols_cic
)

print(f'[+] UNSW named features : {len(unsw_feature_names)}')
print(f'    Sample: {unsw_feature_names[:8]}')
print(f'[+] CIC  named features : {len(cic_feature_names)}')
print(f'    Sample: {cic_feature_names[:8]}')
print('✅ Cell 8 complete — real feature names ready')

[+] UNSW named features : 82
    Sample: ['srcip', 'dstip', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss']
[+] CIC  named features : 78
    Sample: ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min']
✅ Cell 8 complete — real feature names ready


In [29]:
# ============================================================
# CELL 9 — Train Models on UNSW-NB15
# GPU XGBoost + per-sample latency measurement
# Chunked transform — no RAM spike
# ============================================================
def transform_in_chunks(pipeline, X_df, chunk_size=100_000, label=''):
    """Transforms a dataframe in chunks and stacks results."""
    parts = []
    total = len(X_df)
    for start in range(0, total, chunk_size):
        end   = min(start + chunk_size, total)
        chunk = X_df.iloc[start:end]
        parts.append(pipeline.transform(chunk))
        print(f'    {label} {end:,} / {total:,}')
    return sp.vstack(parts) if sp.issparse(parts[0]) else np.vstack(parts)

print('[+] Loading UNSW-NB15...')
df_unsw    = pd.read_csv(UNSW_CLEAN_PATH, low_memory=False)
y_unsw     = df_unsw['label'].astype(int)
X_unsw_raw = df_unsw.drop(
    columns=['label', 'attack_cat', 'dataset_source'], errors='ignore'
)
del df_unsw; gc.collect()

# Apply high-cardinality factorize (must match Cell 6)
for col in pipeline_unsw.high_card_cols:
    if col in X_unsw_raw.columns:
        X_unsw_raw[col] = pd.factorize(X_unsw_raw[col])[0]

X_train_raw, X_test_raw, y_train_u, y_test_u = train_test_split(
    X_unsw_raw, y_unsw, test_size=0.2, stratify=y_unsw, random_state=42
)
del X_unsw_raw, y_unsw; gc.collect()
print(f'[+] Train: {len(X_train_raw):,} | Test: {len(X_test_raw):,}')

print('[+] Transforming train set...')
X_train_u_t = transform_in_chunks(pipeline_unsw, X_train_raw, label='Train')
print('[+] Transforming test set...')
X_test_u_t  = transform_in_chunks(pipeline_unsw, X_test_raw, label='Test')
del X_train_raw, X_test_raw; gc.collect()
print(f'[+] Train: {X_train_u_t.shape} | Test: {X_test_u_t.shape}')

MODELS_UNSW = {
    'random_forest': RandomForestClassifier(
        n_estimators=200, n_jobs=-1, random_state=42
    ),
    'xgboost': XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=8,
        subsample=0.8, colsample_bytree=0.8,
        tree_method=XGB_TREE_METHOD, device=XGB_DEVICE,
        eval_metric='logloss'
    ),
    'logistic_regression': LogisticRegression(max_iter=2000, n_jobs=-1)
}

unsw_metrics        = {}
unsw_trained_models = {}

for name, model in MODELS_UNSW.items():
    print(f'\n[{name}] Training...')
    t0 = time.time()
    model.fit(X_train_u_t, y_train_u)
    train_time = time.time() - t0

    # Per-sample latency benchmark
    n_lat, latencies = min(1000, X_test_u_t.shape[0]), []
    for i in range(n_lat):
        t_s = time.time()
        model.predict(X_test_u_t[i:i+1])
        latencies.append((time.time() - t_s) * 1000)

    preds = model.predict(X_test_u_t)
    acc   = accuracy_score(y_test_u, preds)
    pr, rc, f1, _ = precision_recall_fscore_support(
        y_test_u, preds, average='binary', zero_division=0
    )

    joblib.dump(model, f'{UNSW_MODEL_DIR}/{name}.pkl')
    unsw_trained_models[name] = model
    unsw_metrics[name] = {
        'accuracy':           round(float(acc), 4),
        'precision':          round(float(pr),  4),
        'recall':             round(float(rc),  4),
        'f1':                 round(float(f1),  4),
        'train_time_seconds': round(train_time,  2),
        'latency_ms': {
            'mean': round(float(np.mean(latencies)),           4),
            'p50':  round(float(np.percentile(latencies,50)), 4),
            'p95':  round(float(np.percentile(latencies,95)), 4),
            'p99':  round(float(np.percentile(latencies,99)), 4),
        }
    }
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  F1 Score  : {f1:.4f}')
    print(f'  Train time: {train_time:.1f}s')
    print(f'  Latency   : mean={np.mean(latencies):.3f}ms | '
          f'p95={np.percentile(latencies,95):.3f}ms | '
          f'p99={np.percentile(latencies,99):.3f}ms')

with open(f'{METRICS_DIR}/unsw_nb15_model_metrics.json', 'w') as f:
    json.dump(unsw_metrics, f, indent=4)
print(f'\n✅ Cell 9 complete — UNSW-NB15 models trained and saved')

[+] Loading UNSW-NB15...
[+] Train: 1,647,532 | Test: 411,883
[+] Transforming train set...
    Train 100,000 / 1,647,532
    Train 200,000 / 1,647,532
    Train 300,000 / 1,647,532
    Train 400,000 / 1,647,532
    Train 500,000 / 1,647,532
    Train 600,000 / 1,647,532
    Train 700,000 / 1,647,532
    Train 800,000 / 1,647,532
    Train 900,000 / 1,647,532
    Train 1,000,000 / 1,647,532
    Train 1,100,000 / 1,647,532
    Train 1,200,000 / 1,647,532
    Train 1,300,000 / 1,647,532
    Train 1,400,000 / 1,647,532
    Train 1,500,000 / 1,647,532
    Train 1,600,000 / 1,647,532
    Train 1,647,532 / 1,647,532
[+] Transforming test set...
    Test 100,000 / 411,883
    Test 200,000 / 411,883
    Test 300,000 / 411,883
    Test 400,000 / 411,883
    Test 411,883 / 411,883
[+] Train: (1647532, 82) | Test: (411883, 82)

[random_forest] Training...
  Accuracy  : 0.9949
  F1 Score  : 0.9474
  Train time: 591.8s
  Latency   : mean=47.756ms | p95=67.614ms | p99=80.571ms

[xgboost] Training...

In [32]:
# ============================================================
# CELL 10 — Train Models on CIC-IDS-2017
# Chunked read + transform handles full 3M row dataset
# GPU XGBoost + per-sample latency measurement
# FIXED: label NaN/inf safety + empty chunk guard
# ============================================================
print('[+] Loading + transforming CIC-IDS-2017 in chunks...')

all_X, all_y, rows_done, chunks_skipped = [], [], 0, 0

for chunk in pd.read_csv(CIC_CLEAN_PATH, chunksize=100_000, low_memory=False):

    # ── Fix label column safely ──
    chunk['label'] = pd.to_numeric(chunk['label'], errors='coerce')
    chunk = chunk.dropna(subset=['label'])
    chunk = chunk[chunk['label'].isin([0, 1])]

    # ── Skip empty chunks ──
    if len(chunk) == 0:
        chunks_skipped += 1
        continue

    y_chunk = chunk['label'].astype(int).values
    X_chunk = chunk.drop(
        columns=['label', 'attack_cat', 'dataset_source'], errors='ignore'
    )

    # Apply high-cardinality factorize (must match Cell 7)
    for col in high_card_cols_cic:
        if col in X_chunk.columns:
            X_chunk[col] = pd.factorize(X_chunk[col])[0]

    all_X.append(pipeline_cic.transform(X_chunk))
    all_y.append(y_chunk)
    rows_done += len(chunk)
    print(f'    Processed {rows_done:,} rows...')

print(f'[+] Chunks skipped (bad labels): {chunks_skipped}')

if len(all_X) == 0:
    raise RuntimeError(
        "No valid data after label cleaning. "
        "Re-run Cell 4 to rebuild clean_cic_ids2017.csv"
    )

X_all_cic = sp.vstack(all_X) if sp.issparse(all_X[0]) else np.vstack(all_X)
y_all_cic = np.concatenate(all_y)
del all_X, all_y; gc.collect()
print(f'[+] Full CIC transformed: {X_all_cic.shape}')

X_train_c_t, X_test_c_t, y_train_c, y_test_c = train_test_split(
    X_all_cic, y_all_cic,
    test_size=0.2, stratify=y_all_cic, random_state=42
)
del X_all_cic, y_all_cic; gc.collect()
print(f'[+] Train: {X_train_c_t.shape} | Test: {X_test_c_t.shape}')

MODELS_CIC = {
    'random_forest': RandomForestClassifier(
        n_estimators=200, n_jobs=-1, random_state=42
    ),
    'xgboost': XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=8,
        subsample=0.8, colsample_bytree=0.8,
        tree_method=XGB_TREE_METHOD, device=XGB_DEVICE,
        eval_metric='logloss'
    ),
    'logistic_regression': LogisticRegression(max_iter=2000, n_jobs=-1)
}

cic_metrics        = {}
cic_trained_models = {}

for name, model in MODELS_CIC.items():
    print(f'\n[{name}] Training on CIC-IDS-2017...')
    t0 = time.time()
    model.fit(X_train_c_t, y_train_c)
    train_time = time.time() - t0

    n_lat, latencies = min(1000, X_test_c_t.shape[0]), []
    for i in range(n_lat):
        t_s = time.time()
        model.predict(X_test_c_t[i:i+1])
        latencies.append((time.time() - t_s) * 1000)

    preds = model.predict(X_test_c_t)
    acc   = accuracy_score(y_test_c, preds)
    pr, rc, f1, _ = precision_recall_fscore_support(
        y_test_c, preds, average='binary', zero_division=0
    )

    joblib.dump(model, f'{CIC_MODEL_DIR}/{name}.pkl')
    cic_trained_models[name] = model
    cic_metrics[name] = {
        'accuracy':           round(float(acc), 4),
        'precision':          round(float(pr),  4),
        'recall':             round(float(rc),  4),
        'f1':                 round(float(f1),  4),
        'train_time_seconds': round(train_time,  2),
        'latency_ms': {
            'mean': round(float(np.mean(latencies)),           4),
            'p95':  round(float(np.percentile(latencies, 95)), 4),
            'p99':  round(float(np.percentile(latencies, 99)), 4),
        }
    }
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  F1 Score  : {f1:.4f}')
    print(f'  Train time: {train_time:.1f}s')
    print(f'  Latency   : mean={np.mean(latencies):.3f}ms | '
          f'p95={np.percentile(latencies, 95):.3f}ms')

with open(f'{METRICS_DIR}/cic_ids2017_model_metrics.json', 'w') as f:
    json.dump(cic_metrics, f, indent=4)

print(f'\n✅ Cell 10 complete — CIC-IDS-2017 models trained and saved')

[+] Loading + transforming CIC-IDS-2017 in chunks...
    Processed 100,000 rows...
    Processed 200,000 rows...
    Processed 300,000 rows...
    Processed 400,000 rows...
    Processed 500,000 rows...
    Processed 600,000 rows...
    Processed 700,000 rows...
    Processed 800,000 rows...
    Processed 900,000 rows...
    Processed 1,000,000 rows...
    Processed 1,100,000 rows...
    Processed 1,199,995 rows...
    Processed 1,299,995 rows...
    Processed 1,399,995 rows...
    Processed 1,499,995 rows...
    Processed 1,599,995 rows...
    Processed 1,699,995 rows...
    Processed 1,799,995 rows...
    Processed 1,899,995 rows...
    Processed 1,999,995 rows...
    Processed 2,099,995 rows...
    Processed 2,158,999 rows...
[+] Chunks skipped (bad labels): 0
[+] Full CIC transformed: (2158999, 78)
[+] Train: (1727199, 78) | Test: (431800, 78)

[random_forest] Training on CIC-IDS-2017...
  Accuracy  : 0.9988
  F1 Score  : 0.9962
  Train time: 1364.0s
  Latency   : mean=47.975ms | p

In [1]:
# Run this in a Python terminal or notebook cell
import os
import glob

root = "D:\\AGENTIC-AI-CYBERSECURITY"

print("Searching for CIC metrics file...")
for pattern in [
    "ML/evaluation/metrics/cic_ids2017_model_metrics.json",
    "ML/evaluation/metrics/*.json",
    "**/cic_ids2017_model_metrics.json",
]:
    found = glob.glob(os.path.join(root, pattern), recursive=True)
    if found:
        print(f"✅ FOUND: {pattern}")
        for f in found:
            print(f"   {f}")
    else:
        print(f"❌ Not found: {pattern}")

Searching for CIC metrics file...
✅ FOUND: ML/evaluation/metrics/cic_ids2017_model_metrics.json
   D:\AGENTIC-AI-CYBERSECURITY\ML/evaluation/metrics/cic_ids2017_model_metrics.json
✅ FOUND: ML/evaluation/metrics/*.json
   D:\AGENTIC-AI-CYBERSECURITY\ML/evaluation/metrics\cic_ids2017_evaluation_summary.json
   D:\AGENTIC-AI-CYBERSECURITY\ML/evaluation/metrics\cic_ids2017_model_metrics.json
   D:\AGENTIC-AI-CYBERSECURITY\ML/evaluation/metrics\cross_dataset_evaluation.json
   D:\AGENTIC-AI-CYBERSECURITY\ML/evaluation/metrics\latency_benchmark.json
   D:\AGENTIC-AI-CYBERSECURITY\ML/evaluation/metrics\master_summary.json
   D:\AGENTIC-AI-CYBERSECURITY\ML/evaluation/metrics\unsw_nb15_evaluation_summary.json
   D:\AGENTIC-AI-CYBERSECURITY\ML/evaluation/metrics\unsw_nb15_model_metrics.json
✅ FOUND: **/cic_ids2017_model_metrics.json
   D:\AGENTIC-AI-CYBERSECURITY\ML\evaluation\metrics\cic_ids2017_model_metrics.json


In [33]:
# ============================================================
# CELL 11 — Evaluation Plots
# Confusion matrix + named feature importance + comparison
# Named features fix: real column names instead of Feature_N
# ============================================================
def run_evaluation(models_dict, X_test_t, y_test,
                   feature_names, plot_dir, metrics_path, label):
    os.makedirs(plot_dir, exist_ok=True)
    evaluation = {}

    for key, model in models_dict.items():
        display = MODEL_DISPLAY.get(key, key)
        print(f'  [{label}] {display}...')

        preds = model.predict(X_test_t)
        acc   = accuracy_score(y_test, preds)
        pr, rc, f1, _ = precision_recall_fscore_support(
            y_test, preds, average='binary', zero_division=0
        )
        cm = confusion_matrix(y_test, preds)
        tn, fp, fn, tp = cm.ravel()

        evaluation[key] = {
            'display_name': display, 'dataset': label,
            'metrics': {
                'accuracy':  round(float(acc), 4),
                'precision': round(float(pr),  4),
                'recall':    round(float(rc),  4),
                'f1':        round(float(f1),  4)
            },
            'confusion_matrix': {
                'tn': int(tn), 'fp': int(fp),
                'fn': int(fn), 'tp': int(tp)
            }
        }

        # Confusion matrix
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.title(f'{display} ({label}) — Confusion Matrix')
        plt.xlabel('Predicted'); plt.ylabel('Actual')
        plt.tight_layout()
        plt.savefig(f'{plot_dir}/{key}_confusion_matrix.png', dpi=300)
        plt.close()

        # Feature importance — named
        if hasattr(model, 'feature_importances_'):
            importance = model.feature_importances_
        elif hasattr(model, 'coef_'):
            importance = np.abs(model.coef_[0])
        else:
            continue

        n_feats     = len(importance)
        feat_labels = (
            feature_names
            if (feature_names and len(feature_names) == n_feats)
            else [f'Feature_{i}' for i in range(n_feats)]
        )
        imp_norm = importance / importance.max()
        top_idx  = np.argsort(imp_norm)[-15:][::-1]

        plt.figure(figsize=(10, 6))
        colors = plt.cm.Blues(np.linspace(0.4, 0.9, 15))[::-1]
        plt.barh(
            [feat_labels[i] for i in top_idx],
            imp_norm[top_idx], color=colors
        )
        plt.gca().invert_yaxis()
        plt.xlabel('Normalized Importance Score')
        plt.title(f'{display} ({label}) — Top 15 Feature Importance')
        plt.tight_layout()
        plt.savefig(f'{plot_dir}/{key}_feature_importance.png', dpi=300)
        plt.close()
        print(f'    ✅ Top feature: {feat_labels[top_idx[0]]}')

    # Model comparison
    metric_keys = ['accuracy', 'precision', 'recall', 'f1']
    plt.figure(figsize=(10, 6))
    for key, data in evaluation.items():
        plt.plot(
            metric_keys,
            [data['metrics'][m] for m in metric_keys],
            marker='o', linewidth=2.5,
            label=data['display_name']
        )
    plt.title(f'Model Comparison — {label}')
    plt.xlabel('Metric'); plt.ylabel('Score')
    plt.ylim(0.8, 1.01)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(); plt.tight_layout()
    plt.savefig(f'{plot_dir}/model_comparison.png', dpi=300)
    plt.close()

    with open(metrics_path, 'w') as mf:
        json.dump(evaluation, mf, indent=4)
    print(f'  📊 Plots → {plot_dir}')
    print(f'  📄 Metrics → {metrics_path}')
    return evaluation

print('[UNSW-NB15] Evaluating...')
unsw_eval = run_evaluation(
    unsw_trained_models, X_test_u_t, y_test_u,
    unsw_feature_names, UNSW_PLOT_DIR,
    f'{METRICS_DIR}/unsw_nb15_evaluation_summary.json',
    'UNSW-NB15'
)

print('\n[CIC-IDS-2017] Evaluating...')
cic_eval = run_evaluation(
    cic_trained_models, X_test_c_t, y_test_c,
    cic_feature_names, CIC_PLOT_DIR,
    f'{METRICS_DIR}/cic_ids2017_evaluation_summary.json',
    'CIC-IDS-2017'
)

print('\n✅ Cell 11 complete — all plots generated with named features')

[UNSW-NB15] Evaluating...
  [UNSW-NB15] Random Forest...
    ✅ Top feature: sttl
  [UNSW-NB15] XGBoost...
    ✅ Top feature: sttl
  [UNSW-NB15] Logistic Regression...
    ✅ Top feature: swin
  📊 Plots → d:\AGENTIC-AI-CYBERSECURITY\ML\evaluation\plots\UNSW-NB15
  📄 Metrics → d:\AGENTIC-AI-CYBERSECURITY\ML\evaluation\metrics/unsw_nb15_evaluation_summary.json

[CIC-IDS-2017] Evaluating...
  [CIC-IDS-2017] Random Forest...
    ✅ Top feature: Bwd Packet Length Std
  [CIC-IDS-2017] XGBoost...
    ✅ Top feature: Bwd Packet Length Std
  [CIC-IDS-2017] Logistic Regression...
    ✅ Top feature: Packet Length Variance
  📊 Plots → d:\AGENTIC-AI-CYBERSECURITY\ML\evaluation\plots\CIC-IDS-2017
  📄 Metrics → d:\AGENTIC-AI-CYBERSECURITY\ML\evaluation\metrics/cic_ids2017_evaluation_summary.json

✅ Cell 11 complete — all plots generated with named features


In [34]:
# ============================================================
# CELL 12 — Cross-Dataset Comparison Chart
# Accuracy + F1 side-by-side: UNSW-NB15 vs CIC-IDS-2017
# Publication-ready bar chart
# ============================================================
os.makedirs(CROSS_PLOT_DIR, exist_ok=True)

model_names = [MODEL_DISPLAY[k] for k in unsw_trained_models]
unsw_accs   = [unsw_eval[k]['metrics']['accuracy']  for k in unsw_trained_models]
unsw_f1s    = [unsw_eval[k]['metrics']['f1']         for k in unsw_trained_models]
cic_accs    = [cic_eval[k]['metrics']['accuracy']   for k in cic_trained_models]
cic_f1s     = [cic_eval[k]['metrics']['f1']          for k in cic_trained_models]

x, width = np.arange(len(model_names)), 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, vals_u, vals_c, metric in [
    (axes[0], unsw_accs, cic_accs, 'Accuracy'),
    (axes[1], unsw_f1s,  cic_f1s,  'F1 Score')
]:
    b1 = ax.bar(x - width/2, vals_u, width,
                label='UNSW-NB15',    color='#1f77b4', alpha=0.85)
    b2 = ax.bar(x + width/2, vals_c, width,
                label='CIC-IDS-2017', color='#ff7f0e', alpha=0.85)
    ax.set_xlabel('Model'); ax.set_ylabel(metric)
    ax.set_title(f'{metric} Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=10)
    ax.set_ylim(0.85, 1.005)
    ax.legend(); ax.grid(axis='y', linestyle='--', alpha=0.5)
    for bar in list(b1) + list(b2):
        ax.text(
            bar.get_x() + bar.get_width()/2.,
            bar.get_height() + 0.0005,
            f'{bar.get_height():.4f}',
            ha='center', va='bottom', fontsize=8
        )

plt.suptitle(
    'The Sentinel — Cross-Dataset Performance\nUNSW-NB15 vs CIC-IDS-2017',
    fontsize=13
)
plt.tight_layout()
plt.savefig(f'{CROSS_PLOT_DIR}/cross_dataset_comparison.png', dpi=300)
plt.close()

cross_results = {
    'UNSW-NB15':    {k: unsw_eval[k]['metrics'] for k in unsw_eval},
    'CIC-IDS-2017': {k: cic_eval[k]['metrics']  for k in cic_eval}
}
with open(f'{METRICS_DIR}/cross_dataset_evaluation.json', 'w') as f:
    json.dump(cross_results, f, indent=4)

print(f'✅ Cell 12 complete — cross-dataset chart saved')
print(f'   {CROSS_PLOT_DIR}/cross_dataset_comparison.png')

✅ Cell 12 complete — cross-dataset chart saved
   d:\AGENTIC-AI-CYBERSECURITY\ML\evaluation\plots\CrossDataset/cross_dataset_comparison.png


In [35]:
# ============================================================
# CELL 13 — Latency Benchmark
# Mean / P50 / P95 / P99 distribution per model
# ============================================================
os.makedirs(LATENCY_PLOT_DIR, exist_ok=True)
print('[+] Running latency benchmark (1000 samples per model)...')

latency_results = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (name, model) in enumerate(unsw_trained_models.items()):
    display   = MODEL_DISPLAY[name]
    n_lat     = min(1000, X_test_u_t.shape[0])
    latencies = []
    for i in range(n_lat):
        t0 = time.time()
        model.predict(X_test_u_t[i:i+1])
        latencies.append((time.time() - t0) * 1000)

    lat = np.array(latencies)
    latency_results[name] = {
        'mean_ms':   round(float(np.mean(lat)),           4),
        'median_ms': round(float(np.median(lat)),         4),
        'p95_ms':    round(float(np.percentile(lat,95)),  4),
        'p99_ms':    round(float(np.percentile(lat,99)),  4),
        'min_ms':    round(float(np.min(lat)),            4),
        'max_ms':    round(float(np.max(lat)),            4),
    }
    print(f'  {display}: '
          f'mean={np.mean(lat):.3f}ms | '
          f'p50={np.median(lat):.3f}ms | '
          f'p95={np.percentile(lat,95):.3f}ms | '
          f'p99={np.percentile(lat,99):.3f}ms')

    axes[idx].hist(lat, bins=50, color='steelblue', alpha=0.8, edgecolor='white')
    axes[idx].axvline(np.mean(lat),           color='red',    ls='--', lw=1.5,
                      label=f'Mean: {np.mean(lat):.2f}ms')
    axes[idx].axvline(np.percentile(lat,95),  color='orange', ls='--', lw=1.5,
                      label=f'P95:  {np.percentile(lat,95):.2f}ms')
    axes[idx].axvline(np.percentile(lat,99),  color='purple', ls='--', lw=1.5,
                      label=f'P99:  {np.percentile(lat,99):.2f}ms')
    axes[idx].set_title(f'{display}\nLatency Distribution')
    axes[idx].set_xlabel('Latency (ms)')
    axes[idx].set_ylabel('Count')
    axes[idx].legend(fontsize=8)

plt.suptitle('Inference Latency — UNSW-NB15 (n=1000 samples)', y=1.02)
plt.tight_layout()
plt.savefig(f'{LATENCY_PLOT_DIR}/latency_distribution.png', dpi=300)
plt.close()

with open(f'{METRICS_DIR}/latency_benchmark.json', 'w') as f:
    json.dump(latency_results, f, indent=4)
print(f'✅ Cell 13 complete — latency benchmark saved')

[+] Running latency benchmark (1000 samples per model)...
  Random Forest: mean=93.600ms | p50=100.910ms | p95=117.492ms | p99=134.758ms
  XGBoost: mean=1.411ms | p50=1.141ms | p95=2.495ms | p99=3.006ms
  Logistic Regression: mean=0.113ms | p50=0.000ms | p95=1.003ms | p99=1.162ms
✅ Cell 13 complete — latency benchmark saved


In [36]:
# ============================================================
# CELL 14 — Master Summary Table
# All models, both datasets, all metrics in one view
# ============================================================
print('\n' + '='*78)
print('  THE SENTINEL — MASTER RESULTS SUMMARY')
print('='*78)
print(f'{"Model":<25} {"Dataset":<18} '
      f'{"Accuracy":>10} {"Precision":>11} {"Recall":>8} {"F1":>8}')
print('-'*78)

for key in ['random_forest', 'xgboost', 'logistic_regression']:
    d = MODEL_DISPLAY[key]
    u = unsw_eval[key]['metrics']
    c = cic_eval[key]['metrics']
    print(f'{d:<25} {"UNSW-NB15":<18} '
          f'{u["accuracy"]:>10.4f} {u["precision"]:>11.4f} '
          f'{u["recall"]:>8.4f} {u["f1"]:>8.4f}')
    print(f'{"":<25} {"CIC-IDS-2017":<18} '
          f'{c["accuracy"]:>10.4f} {c["precision"]:>11.4f} '
          f'{c["recall"]:>8.4f} {c["f1"]:>8.4f}')
    print()

print('='*78)
print('\nLatency Summary (UNSW-NB15 | per-sample inference, n=1000):')
print(f'{"Model":<25} {"Mean":>10} {"P50":>10} {"P95":>10} {"P99":>10}')
print('-'*65)
for key, lat in latency_results.items():
    d = MODEL_DISPLAY[key]
    print(f'{d:<25} '
          f'{lat["mean_ms"]:>9.3f}ms '
          f'{lat["median_ms"]:>9.3f}ms '
          f'{lat["p95_ms"]:>9.3f}ms '
          f'{lat["p99_ms"]:>9.3f}ms')

master = {
    'UNSW-NB15':     unsw_eval,
    'CIC-IDS-2017':  cic_eval,
    'latency':       latency_results,
    'cross_dataset': cross_results
}
with open(f'{METRICS_DIR}/master_summary.json', 'w') as f:
    json.dump(master, f, indent=4)

print(f'\n📄 master_summary.json saved to {METRICS_DIR}/')
print('✅ Cell 14 complete')


  THE SENTINEL — MASTER RESULTS SUMMARY
Model                     Dataset              Accuracy   Precision   Recall       F1
------------------------------------------------------------------------------
Random Forest             UNSW-NB15              0.9949      0.9476   0.9473   0.9474
                          CIC-IDS-2017           0.9988      0.9971   0.9953   0.9962

XGBoost                   UNSW-NB15              0.9942      0.9457   0.9329   0.9393
                          CIC-IDS-2017           0.9992      0.9976   0.9974   0.9975

Logistic Regression       UNSW-NB15              0.9881      0.8436   0.9263   0.8830
                          CIC-IDS-2017           0.9721      0.9653   0.8506   0.9043


Latency Summary (UNSW-NB15 | per-sample inference, n=1000):
Model                           Mean        P50        P95        P99
-----------------------------------------------------------------
Random Forest                93.600ms   100.910ms   117.492ms   134.758ms
XGBo

In [37]:
# ============================================================
# CELL 15 — Save Inference Registry
# Bundles all pipelines + models for backend use
# Keys: unsw_random_forest, unsw_xgboost, unsw_logistic_regression
#       cic_random_forest,  cic_xgboost,  cic_logistic_regression
# ============================================================
registry = {}

for name, model in unsw_trained_models.items():
    registry[f'unsw_{name}'] = {
        'pipeline': pipeline_unsw,
        'model':    model,
        'dataset':  'UNSW-NB15'
    }

for name, model in cic_trained_models.items():
    registry[f'cic_{name}'] = {
        'pipeline': pipeline_cic,
        'model':    model,
        'dataset':  'CIC-IDS-2017'
    }

joblib.dump(registry, INFERENCE_PATH)

print(f'[+] Registry keys saved:')
for k in registry:
    print(f'    {k}')
print(f'\n✅ Cell 15 complete — registry saved to {INFERENCE_PATH}')
print('\n' + '='*60)
print('  🛡️  THE SENTINEL — FULL PIPELINE COMPLETE')
print('='*60)
print(f'  Datasets  : UNSW-NB15 + CIC-IDS-2017')
print(f'  Models    : Random Forest · XGBoost · Logistic Regression')
print(f'  GPU       : {"XGBoost used CUDA (GT 1060)" if GPU_AVAILABLE else "CPU only"}')
print(f'  Features  : Named (not Feature_N)')
print(f'  Latency   : Measured P50 / P95 / P99')
print(f'  Plots     : ML/evaluation/plots/')
print(f'  Metrics   : ML/evaluation/metrics/')
print(f'  Registry  : {INFERENCE_PATH}')
print('='*60)

[+] Registry keys saved:
    unsw_random_forest
    unsw_xgboost
    unsw_logistic_regression
    cic_random_forest
    cic_xgboost
    cic_logistic_regression

✅ Cell 15 complete — registry saved to d:\AGENTIC-AI-CYBERSECURITY\ML\inference_models\inference_registry.pkl

  🛡️  THE SENTINEL — FULL PIPELINE COMPLETE
  Datasets  : UNSW-NB15 + CIC-IDS-2017
  Models    : Random Forest · XGBoost · Logistic Regression
  GPU       : CPU only
  Features  : Named (not Feature_N)
  Latency   : Measured P50 / P95 / P99
  Plots     : ML/evaluation/plots/
  Metrics   : ML/evaluation/metrics/
  Registry  : d:\AGENTIC-AI-CYBERSECURITY\ML\inference_models\inference_registry.pkl
